<a href="https://colab.research.google.com/github/vtminc1000/Analisis-Sentimen/blob/main/Uji1kalimat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas openpyxl transformers torch vaderSentiment textblob deep-translator matplotlib -q

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

from google.colab import files
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from deep_translator import GoogleTranslator

In [ ]:
uploaded = files.upload()

In [ ]:
uploaded = files.upload()

In [ ]:
uploaded = files.upload()

In [ ]:
df = pd.read_excel('Data_Inite.xlsx')
positive = pd.read_csv('positive.tsv.txt', sep='\t')
negative = pd.read_csv('negative.tsv.txt', sep='\t')

print(df.head())
print(positive.head())
print(negative.head())

In [ ]:
data_long = []

for _, row in df.iterrows():
    data_long.append({
        'responden_id': row['responden_id'],
        'tahap': 'pre',
        'pertanyaan': 'q1',
        'teks': row['pre_q1'],
        'label_asli': row['pre_q1_label']
    })
    data_long.append({
        'responden_id': row['responden_id'],
        'tahap': 'pre',
        'pertanyaan': 'q2',
        'teks': row['pre_q2'],
        'label_asli': row['pre_q2_label']
    })
    data_long.append({
        'responden_id': row['responden_id'],
        'tahap': 'post',
        'pertanyaan': 'q1',
        'teks': row['post_q1'],
        'label_asli': row['post_q1_label']
    })
    data_long.append({
        'responden_id': row['responden_id'],
        'tahap': 'post',
        'pertanyaan': 'q2',
        'teks': row['post_q2'],
        'label_asli': row['post_q2_label']
    })

data_long = pd.DataFrame(data_long)
data_long.head(10)

In [ ]:
kalimat_uji = "untuk pembangkit listrik tenaga nuklir"
label_manual = 1

In [ ]:
df_uji = pd.DataFrame({
    'teks': [kalimat_uji],
    'label_manual': [label_manual]
})

df_uji

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_uji['teks_clean'] = df_uji['teks'].apply(clean_text)
df_uji

In [ ]:
lexicon_df = pd.concat([positive, negative], ignore_index=True)
inset_dict = dict(zip(lexicon_df['word'], lexicon_df['weight']))
list(inset_dict.items())[:10]

In [ ]:
def inset_predict(text):
    score = 0
    words = str(text).split()

    for word in words:
        if word in inset_dict:
            score += inset_dict[word]

    if score > 0:
        return 1
    elif score < 0:
        return -1
    else:
        return 0

In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

In [ ]:
def indobert_predict(text):
    result = classifier(str(text))[0]
    label = result['label'].lower()

    if label == 'negative':
        return -1
    elif label == 'neutral':
        return 0
    else:
        return 1

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def translate_text(text):
    try:
        return GoogleTranslator(source='id', target='en').translate(str(text))
    except:
        return str(text)

def vader_predict(text):
    score = analyzer.polarity_scores(str(text))['compound']
    if score >= 0.05:
        return 1
    elif score <= -0.05:
        return -1
    else:
        return 0

In [ ]:
def textblob_predict(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0:
        return 1
    elif polarity < 0:
        return -1
    else:
        return 0

In [ ]:
df_uji['inset'] = df_uji['teks_clean'].apply(inset_predict)
df_uji['indobert'] = df_uji['teks_clean'].apply(indobert_predict)
df_uji['teks_en'] = df_uji['teks_clean'].apply(translate_text)
df_uji['vader'] = df_uji['teks_en'].apply(vader_predict)
df_uji['textblob'] = df_uji['teks_en'].apply(textblob_predict)

df_uji

In [ ]:
df_plot = pd.DataFrame({
    'Metode': ['Manual', 'InSet', 'IndoBERT', 'VADER', 'TextBlob'],
    'Nilai': [
        df_uji.loc[0, 'label_manual'],
        df_uji.loc[0, 'inset'],
        df_uji.loc[0, 'indobert'],
        df_uji.loc[0, 'vader'],
        df_uji.loc[0, 'textblob']
    ]
})

df_plot

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(df_plot['Metode'], df_plot['Nilai'])

plt.title('Perbandingan Sentimen pada 1 Kalimat Uji')
plt.xlabel('Metode')
plt.ylabel('Nilai Sentimen (-1, 0, 1)')
plt.axhline(0, color='black', linewidth=0.8)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(df_plot['Metode'], df_plot['Nilai'])

plt.title('Perbandingan Sentimen pada 1 Kalimat Uji')
plt.xlabel('Metode')
plt.ylabel('Nilai Sentimen (-1, 0, 1)')
plt.axhline(0, color='black', linewidth=0.8)

plt.savefig('grafik_1_kalimat_uji.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_uji.to_excel('hasil_1_kalimat_uji.xlsx', index=False)
df_plot.to_excel('ringkasan_1_kalimat_uji.xlsx', index=False)

files.download('hasil_1_kalimat_uji.xlsx')
files.download('ringkasan_1_kalimat_uji.xlsx')
files.download('grafik_1_kalimat_uji.png')